In [ ]:
import multiprocessing
import time
import os
import sys

# ._internal_worker は、あなたが提示した pipeline_full.py の
# process_image_subdirectory 関数を import している想定です
from pipeline_full import process_image_subdirectory

def run(base_dir, model_path, target_folders, area_threshold=50000, max_workers=1):
    """
    画像処理パイプラインを実行します。
    
    Args:
        max_workers (int): 
            2以上: 指定したワーカー数で並列実行 (マルチプロセス)
            1以下: 逐次実行 (シングルプロセス・デバッグ/比較用)
    """
    
    print(f"処理を開始します... 対象: {target_folders}")
    overall_start_time = time.time()

    # max_workers が 2 以上の場合のみマルチプロセス（Pool）を使用する
    if max_workers > 1:
        print(f"並列処理を開始します (最大ワーカー数: {max_workers})")
        try:
            multiprocessing.set_start_method('spawn', force=True)
            print("マルチプロセスの開始メソッドを 'spawn' に設定しました。")
        except RuntimeError:
            # 既に設定されている場合は何もしない
            pass 

        # Poolに渡すための引数リストを作成
        tasks = [
            (base_dir, folder, model_path, area_threshold) for folder in target_folders
        ]

        # マルチプロセスで実行
        with multiprocessing.Pool(processes=max_workers) as pool:
            pool.starmap(process_image_subdirectory, tasks)
    
    else:
        # max_workersが 1 以下の場合は、単純な for ループで逐次実行する
        print("逐次処理（シングルプロセス）を開始します。")
        for folder in target_folders:
            # process_image_subdirectory を直接呼び出す
            process_image_subdirectory(
                base_data_dir=base_dir, 
                sub_folder=folder, 
                model_path=model_path, 
                area_threshold=area_threshold
            )

    overall_end_time = time.time()
    print(f"\nすべての処理が完了しました。 (総所要時間: {overall_end_time - overall_start_time:.2f}秒)")
    
    run(
    base_dir="/home/data/1104_test",
    model_path='/home/YOLO/0708_maesyori/datasets/train/weights/best.pt',
    target_folders=['A', 'B', 'C'],
    max_workers=1
)

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


In [6]:
run(
    base_dir="/home/data/1104_test",
    model_path='/home/YOLO/0708_maesyori/datasets/train/weights/best.pt',
    target_folders=['A', 'B', 'C'],
    max_workers=1
)

処理を開始します... 対象: ['A', 'B', 'C']
逐次処理（シングルプロセス）を開始します。
--- [66882] 処理開始: A ---
[66882] ステップ1 開始: /home/data/1104_test/org/A

0: 640x480 52 shiitakes, 3.1ms
Speed: 1.5ms preprocess, 3.1ms inference, 17.9ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 53 shiitakes, 3.1ms
Speed: 1.2ms preprocess, 3.1ms inference, 18.3ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 41 shiitakes, 3.2ms
Speed: 1.2ms preprocess, 3.2ms inference, 14.3ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 41 shiitakes, 3.1ms
Speed: 1.3ms preprocess, 3.1ms inference, 14.2ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 55 shiitakes, 3.2ms
Speed: 1.2ms preprocess, 3.2ms inference, 18.9ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 41 shiitakes, 3.3ms
Speed: 1.2ms preprocess, 3.3ms inference, 14.3ms postprocess per image at shape (1, 3, 640, 480)

0: 640x480 43 shiitakes, 3.2ms
Speed: 1.2ms preprocess, 3.2ms inference, 14.9ms postprocess per image 

In [7]:
# (Jupyter Notebook側のセル)
# import concurrent.futures # ← インポート不要
from pipeline_full import process_image_subdirectory # 関数をインポート
import time

# ... (base_data_dirやmodel_pathなどの設定) ...

sub_folders = ['A', 'B', 'C'] # 処理対象フォルダ
# max_workers = 4 # ← 不要

print("処理を開始します (シングルプロセス)")
start_time_single = time.time()

# ▼▼▼ ここを単純なforループに変更 ▼▼▼
for folder in sub_folders:
    try:
        # 関数を直接呼び出す
        process_image_subdirectory(
            base_dir="/home/data/1104_test",
            model_path='/home/YOLO/0708_maesyori/datasets/train/weights/best.pt',
            target_folders=['A', 'B', 'C'],
        )
    except Exception as e:
        print(f"フォルダ {folder} の処理中にエラーが発生しました: {e}")
# ▲▲▲ ここまで ▲▲▲

end_time_single = time.time()
print(f"全処理が完了しました (シングルプロセス)。総所要時間: {end_time_single - start_time_single:.2f}秒")

処理を開始します (シングルプロセス)
フォルダ A の処理中にエラーが発生しました: process_image_subdirectory() got an unexpected keyword argument 'base_dir'
フォルダ B の処理中にエラーが発生しました: process_image_subdirectory() got an unexpected keyword argument 'base_dir'
フォルダ C の処理中にエラーが発生しました: process_image_subdirectory() got an unexpected keyword argument 'base_dir'
全処理が完了しました (シングルプロセス)。総所要時間: 0.00秒
